# Clasificador OpenAI gpt-5-mini
Clasifica el dataset de preguntas de red usando la API de OpenAI.

In [37]:
pip install openai pandas tqdm

Note: you may need to restart the kernel to use updated packages.


## Configuración

In [46]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../").resolve() / ".env")

API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-5-mini"
INPUT_CSV = "requirements_questions_v2.csv"
OUTPUT_CSV = "requirements_classified_gpt5.csv"
SAMPLE = None  # usa un int para prueba rápida

print(f"API_KEY cargada: {'OK' if API_KEY else 'NO ENCONTRADA'}")
print(f"Modelo: {MODEL}")

API_KEY cargada: OK
Modelo: gpt-5-mini


## Setup

In [ ]:
import json
import pandas as pd
from tqdm.notebook import tqdm
from openai import OpenAI

client = OpenAI(api_key=API_KEY)

CAT_LIST = ["ROUTING", "SECURITY", "QOS", "CONNECTIVITY", "MONITORING", "GENERAL"]
DEBUG_EACH_QUESTION = True
REASONING_EFFORT = "low"

SYSTEM_PROMPT = """You are a Cisco network expert and dataset annotator.
Classify each question into exactly one category from this taxonomy:

ROUTING, SECURITY, QOS, CONNECTIVITY, MONITORING, GENERAL

Taxonomy (brief):
- ROUTING: route calculation/selection and control-plane routing protocols (OSPF, BGP, EIGRP, redistribution, route-maps for routing policy).
- SECURITY: access control and protection mechanisms (ACLs, firewall, VPN/IPsec, AAA, authentication/authorization, segmentation for security intent).
- QOS: traffic classification, marking, policing, shaping, queuing, congestion management/avoidance, and service policies.
- CONNECTIVITY: interface/link bring-up, Layer 2/Layer 3 reachability, addressing, VLAN/trunks, tunnels/overlay setup, adjacency/neighbor establishment issues.
- MONITORING: telemetry, observability, health/performance measurement, logs, SNMP, NetFlow/IPFIX, IP SLA, troubleshooting visibility commands.
- GENERAL: conceptual/vendor-agnostic networking questions that do not fit the above technical domains.

Decision rules:
- Choose the SINGLE most relevant category based on primary intent.
- If multiple domains appear, label by what the user is mainly asking to configure/fix/verify.
- Use GENERAL only as a last resort.
- Respond with a JSON block only. No extra text.

Response format:
```json
{\"category\": \"CATEGORY_NAME\"}
```"""

print("Setup OK")

Setup OK


## Funciones

In [41]:
def parse_json_from_response(text: str) -> dict | None:
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    match = re.search(r"\{[^{}]*\"category\"[^{}]*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    return None


def debug_output(question: str, raw: str, pred: str):
    if not DEBUG_EACH_QUESTION:
        return
    print(
        f"[QUESTION]\n{question}\n[RAW]\n{raw if raw else '<VACÍO>'}\n[PRED]\n{pred}\n"
    )


def classify(question: str) -> str:
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": question},
            ],
            max_completion_tokens=512,
            reasoning_effort=REASONING_EFFORT,
        )
        raw = (response.choices[0].message.content or "").strip()

        parsed = parse_json_from_response(raw)
        if parsed and parsed.get("category") in CAT_LIST:
            pred = parsed["category"]
            debug_output(question, raw, pred)
            return pred

        for cat in CAT_LIST:
            if cat in raw.upper():
                debug_output(question, raw, cat)
                return cat

        debug_output(question, raw, "UNKNOWN")
        return "UNKNOWN"
    except Exception as e:
        print(f"\n[UNKNOWN] QUESTION:\n{question}\n[ERROR]: {e}\n")
        return "UNKNOWN"

print("Funciones OK")

Funciones OK


## Cargar dataset

In [43]:
df = pd.read_csv(INPUT_CSV)
if "id" not in df.columns:
    df.insert(0, "id", range(len(df)))

if SAMPLE:
    df = df.sample(n=SAMPLE, random_state=42).reset_index(drop=True)
    print(f"Modo prueba: {SAMPLE} registros")

print(f"Total: {len(df)} preguntas")
df[["id", "question"]].head()

Total: 4570 preguntas


,id,question
0,0,How can we monitor and store download time wit...
1,1,How can we ensure a network with broader reach...
2,2,How can you ensure that IP SLAs measurements a...
3,3,How do I configure the IP Service Level Agreem...
4,4,How do I enable the IP SLAs Responder on my Ci...


## Clasificar

In [44]:
results: dict[int, str] = {}

all_ids = [int(x) for x in df["id"].tolist()]
print(f"Registros a clasificar: {len(all_ids)}")

for count, row_id in enumerate(tqdm(all_ids, total=len(all_ids)), start=1):
    row = df[df["id"] == row_id].iloc[0]
    question = str(row["question"])

    if DEBUG_EACH_QUESTION:
        print(f"\n--- [{count}/{len(all_ids)}] id={row_id} ---")

    results[row_id] = classify(question)

print("Clasificación completa")

Registros a clasificar: 4570


  0%|          | 0/4570 [00:00<?, ?it/s]


--- [1/4570] id=0 ---
[QUESTION]
How can we monitor and store download time within the Cisco device and configure IP and application layer options for optimal performance?
[RAW]
{"category": "MONITORING"}
[PRED]
MONITORING


--- [2/4570] id=1 ---
[QUESTION]
How can we ensure a network with broader reach and accurate representation of end-user experience while maintaining ease of deployment?
[RAW]
{"category": "MONITORING"}
[PRED]
MONITORING


--- [3/4570] id=2 ---
[QUESTION]
How can you ensure that IP SLAs measurements are continuous, reliable, and predictable?
[RAW]
{"category":"MONITORING"}
[PRED]
MONITORING


--- [4/4570] id=3 ---
[QUESTION]
How do I configure the IP Service Level Agreement (SLA) feature on an ASR 920 platform to measure network performance?
[RAW]
{"category": "MONITORING"}
[PRED]
MONITORING


--- [5/4570] id=4 ---
[QUESTION]
How do I enable the IP SLAs Responder on my Cisco device?
[RAW]
{"category": "MONITORING"}
[PRED]
MONITORING


--- [6/4570] id=5 ---
[QUESTIO

## Exportar y revisar resultados

In [47]:
df["category"] = [results.get(int(row_id), "UNKNOWN") for row_id in df["id"]]
df.to_csv(OUTPUT_CSV, index=False)
print(f"Dataset guardado: {OUTPUT_CSV}")

unknown_count = int((df["category"] == "UNKNOWN").sum())
print(f"UNKNOWN: {unknown_count}")

print("\n=== DISTRIBUCIÓN FINAL ===")
display(df["category"].value_counts().to_frame())

df[["id", "question", "category"]].head(10)

Dataset guardado: requirements_classified_gpt5.csv
UNKNOWN: 0

=== DISTRIBUCIÓN FINAL ===


,count
category,
SECURITY,1269
ROUTING,1161
MONITORING,777
QOS,652
CONNECTIVITY,612
GENERAL,99


,id,question,category
0,0,How can we monitor and store download time wit...,MONITORING
1,1,How can we ensure a network with broader reach...,MONITORING
2,2,How can you ensure that IP SLAs measurements a...,MONITORING
3,3,How do I configure the IP Service Level Agreem...,MONITORING
4,4,How do I enable the IP SLAs Responder on my Ci...,MONITORING
5,5,How can I configure an IP SLAs Responder to li...,MONITORING
6,6,What is the recommended configuration to minim...,MONITORING
7,7,How can we improve the accuracy of network act...,MONITORING
8,8,How can we configure the Cisco router to send ...,MONITORING
9,9,Can we configure IP SLAs to maintain two hours...,MONITORING
